In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC, SYN = ROOT / "data" / "processed", ROOT / "data" / "synthetic"

raw = (pd.read_csv(PROC / "engineered_features_raw.csv")
         .merge(pd.read_csv(PROC / "household_split.csv"), on="household_id")
         .merge(pd.read_csv(SYN / "households.csv")[["household_id", "urban_rural"]], on="household_id"))

FIXED = ["housing", "loan_repayment", "school_fees"]                     # commitments: set aside exactly
VARIABLE = ["food_groceries", "transport", "utilities", "airtime_data",
            "mobile_money_transfers", "entertainment"]                    # everyday money: split by the model

raw["variable_total"] = raw[VARIABLE].sum(axis=1)
raw["available"] = (raw["total_income"] - raw[FIXED].sum(axis=1) - raw["savings"]).clip(lower=0)
raw["is_formal"] = (raw["income_tier"] == "formal_salaried").astype(int)
raw["urban"] = (raw["urban_rural"] == "urban").astype(int)
raw["peri_urban"] = (raw["urban_rural"] == "peri_urban").astype(int)

# Learn only from Stable household-months: "what healthy households like you do".
stable = raw[(raw.true_state == "Stable") & (raw.variable_total > 0)].copy()
for c in VARIABLE:
    stable[f"{c}_share"] = stable[c] / stable["variable_total"]

X_COLS = ["total_income", "available", "household_size", "num_dependants", "is_formal", "urban", "peri_urban"]
Y_COLS = [f"{c}_share" for c in VARIABLE]
tr, te = stable[stable.split == "train"], stable[stable.split == "test"]

split_model = RandomForestRegressor(n_estimators=300, min_samples_leaf=10, random_state=0)
split_model.fit(tr[X_COLS], tr[Y_COLS])

pred = split_model.predict(te[X_COLS])
pred = pred / pred.sum(axis=1, keepdims=True)                 # shares must add up to 100%
one_size = np.tile(tr[Y_COLS].mean().values, (len(te), 1))   # the simple alternative

def kes_error(p):
    """Average error in KES per category, on Stable households the model never saw."""
    return (np.abs(p - te[Y_COLS].values) * te[["variable_total"]].values).mean(axis=0)

print(f"learned from {len(tr)} Stable household-months, tested on {len(te)}")
pd.DataFrame({"one-size-fits-all (KES off)": kes_error(one_size).round(0),
              "Random Forest (KES off)": kes_error(pred).round(0)}, index=VARIABLE)

learned from 974 Stable household-months, tested on 215


,one-size-fits-all (KES off),Random Forest (KES off)
food_groceries,1845.0,1806.0
transport,1405.0,1476.0
utilities,858.0,518.0
airtime_data,638.0,528.0
mobile_money_transfers,1540.0,1939.0
entertainment,1009.0,888.0


In [2]:
middle = 0.5 * pred + 0.5 * one_size
table = pd.DataFrame({"one-size-fits-all": kes_error(one_size).round(0),
                      "Random Forest": kes_error(pred).round(0),
                      "half and half": kes_error(middle).round(0)}, index=VARIABLE)
table.loc["TOTAL"] = table.sum()
table


,one-size-fits-all,Random Forest,half and half
food_groceries,1845.0,1806.0,1721.0
transport,1405.0,1476.0,1413.0
utilities,858.0,518.0,643.0
airtime_data,638.0,528.0,575.0
mobile_money_transfers,1540.0,1939.0,1728.0
entertainment,1009.0,888.0,941.0
TOTAL,7295.0,7155.0,7021.0


In [3]:
ONE_SIZE = tr[Y_COLS].mean().values
SAVINGS_RATE = {"Stable": 0.10, "Strained": 0.07, "Critical": 0.05}

def recommend_budget(income, fixed, household_size=1, num_dependants=0, formal_job=False,
                     urban_rural="urban", state="Stable", goal_per_month=0, last_month=None):
    """Monthly limits per category. fixed = {"housing": ..., "loan_repayment": ..., "school_fees": ...}.
    last_month = {"limits": {...}, "spent": {...}} to apply the overspend rule."""
    notes = []
    fixed = {c: float(fixed.get(c, 0)) for c in FIXED}
    fixed_total = sum(fixed.values())
    savings = max(SAVINGS_RATE[state] * income, goal_per_month)
    everyday = income - fixed_total - savings
    if everyday < 0:
        savings = max(0.0, income - fixed_total)
        everyday = income - fixed_total - savings
        notes.append("Your commitments leave little or nothing for everyday spending; savings reduced to fit.")
    if everyday <= 0:
        notes.append("Your commitments are more than your income. Start by reviewing rent or loan repayments.")
        everyday = 0.0

    x = pd.DataFrame([{"total_income": income, "available": max(0.0, income - fixed_total - savings),
                       "household_size": household_size, "num_dependants": num_dependants,
                       "is_formal": int(formal_job), "urban": int(urban_rural == "urban"),
                       "peri_urban": int(urban_rural == "peri_urban")}])
    share = 0.5 * split_model.predict(x[X_COLS])[0] + 0.5 * ONE_SIZE
    share = share / share.sum()
    limits = {c: everyday * s for c, s in zip(VARIABLE, share)}

    if last_month:
        for c in VARIABLE:
            over = last_month["spent"].get(c, 0) - last_month["limits"].get(c, float("inf"))
            if over > 0:
                cut = min(0.5 * over, limits[c])
                limits[c] -= cut
                savings += cut
                notes.append(f"You went KES {round(over):,} over on {c} last month, so this month's limit is "
                             f"KES {round(cut):,} lower and that goes to savings.")

    budget = {**{c: round(v) for c, v in fixed.items()}, **{c: round(v) for c, v in limits.items()},
              "savings": round(savings)}
    return {"income": round(income), "budget": budget, "total": sum(budget.values()), "notes": notes}

# 1) A family earner, Stable vs Strained: savings and everyday limits shift.
fam = dict(income=25300, fixed={"housing": 6000}, household_size=4, num_dependants=2, urban_rural="peri_urban")
print("Stable:  ", recommend_budget(**fam, state="Stable"))
print("Strained:", recommend_budget(**fam, state="Strained"))

# 2) A student on KES 12,000 (allowance + HELB spread monthly), hostel KES 4,000.
student_b = recommend_budget(12000, {"housing": 4000}, state="Strained")
print("\nStudent: ", student_b)

# 3) Next month for the student, after overspending on entertainment by KES 600.
spent = dict(student_b["budget"]); spent["entertainment"] += 600
print("\nStudent, next month:", recommend_budget(12000, {"housing": 4000}, state="Strained",
      last_month={"limits": student_b["budget"], "spent": spent}))

Stable:   {'income': 25300, 'budget': {'housing': 6000, 'loan_repayment': 0, 'school_fees': 0, 'food_groceries': 8571, 'transport': 2133, 'utilities': 1827, 'airtime_data': 1005, 'mobile_money_transfers': 2072, 'entertainment': 1162, 'savings': 2530}, 'total': 25300, 'notes': []}
Strained: {'income': 25300, 'budget': {'housing': 6000, 'loan_repayment': 0, 'school_fees': 0, 'food_groceries': 8853, 'transport': 2285, 'utilities': 1908, 'airtime_data': 1060, 'mobile_money_transfers': 2187, 'entertainment': 1236, 'savings': 1771}, 'total': 25300, 'notes': []}

Student:  {'income': 12000, 'budget': {'housing': 4000, 'loan_repayment': 0, 'school_fees': 0, 'food_groceries': 3334, 'transport': 901, 'utilities': 924, 'airtime_data': 462, 'mobile_money_transfers': 934, 'entertainment': 605, 'savings': 840}, 'total': 12000, 'notes': []}

Student, next month: {'income': 12000, 'budget': {'housing': 4000, 'loan_repayment': 0, 'school_fees': 0, 'food_groceries': 3334, 'transport': 901, 'utilities': 

In [4]:
TRIM_FUN = {"Stable": 0.0, "Strained": 0.25, "Critical": 0.50}   # share of entertainment moved to savings

def recommend_budget(income, fixed, household_size=1, num_dependants=0, formal_job=False,
                     urban_rural="urban", state="Stable", goal_per_month=0, last_month=None):
    notes = []
    fixed = {c: float(fixed.get(c, 0)) for c in FIXED}
    fixed_total = sum(fixed.values())
    savings = max(0.10 * income, goal_per_month)
    everyday = income - fixed_total - savings
    if everyday < 0:
        savings = max(0.0, income - fixed_total)
        everyday = income - fixed_total - savings
        notes.append("Your commitments leave little or nothing for everyday spending; savings reduced to fit.")
    if everyday <= 0:
        notes.append("Your commitments are more than your income. Start by reviewing rent or loan repayments.")
        everyday = 0.0

    x = pd.DataFrame([{"total_income": income, "available": max(0.0, income - fixed_total - savings),
                       "household_size": household_size, "num_dependants": num_dependants,
                       "is_formal": int(formal_job), "urban": int(urban_rural == "urban"),
                       "peri_urban": int(urban_rural == "peri_urban")}])
    share = 0.5 * split_model.predict(x[X_COLS])[0] + 0.5 * ONE_SIZE
    share = share / share.sum()
    limits = {c: everyday * s for c, s in zip(VARIABLE, share)}

    if TRIM_FUN[state] > 0:
        trim = limits["entertainment"] * TRIM_FUN[state]
        limits["entertainment"] -= trim
        savings += trim
        notes.append(f"Because things are {state.lower()}, entertainment is KES {round(trim):,} lower "
                     "and that goes to savings to rebuild your buffer.")

    if last_month:
        for c in VARIABLE:
            over = last_month["spent"].get(c, 0) - last_month["limits"].get(c, float("inf"))
            if over > 0:
                cut = min(0.5 * over, limits[c])
                limits[c] -= cut
                savings += cut
                notes.append(f"You went KES {round(over):,} over on {c} last month, so this month's limit is "
                             f"KES {round(cut):,} lower and that goes to savings.")

    budget = {**{c: round(v) for c, v in fixed.items()}, **{c: round(v) for c, v in limits.items()},
              "savings": round(savings)}
    return {"income": round(income), "budget": budget, "total": sum(budget.values()), "notes": notes}

for state in ["Stable", "Strained", "Critical"]:
    b = recommend_budget(**fam, state=state)["budget"]
    print(f"{state:9} entertainment {b['entertainment']:>6,}   savings {b['savings']:>6,}")

Stable    entertainment  1,162   savings  2,530
Strained  entertainment    872   savings  2,821
Critical  entertainment    581   savings  3,111


In [5]:
import sys
sys.path.insert(0, str(ROOT / "backend"))
import affordability as aff

rows = []
last = raw[(raw.split == "test") & (raw.month == 12) & (raw.true_state != "Stable") & (raw.total_income > 0)]
for _, r in last.iterrows():
    hist = raw[raw.household_id == r.household_id].sort_values("month")
    months = hist[aff.CATS + aff.INCOME].to_dict("records")
    b = recommend_budget(r.total_income, {c: r[c] for c in FIXED}, r.household_size, r.num_dependants,
                         r.income_tier == "formal_salaried", r.urban_rural, r.true_state)["budget"]
    follow = [dict(m) for m in months]
    for m in follow[-3:]:                      # pretend they followed the budget for 3 months
        for c in aff.CATS:
            m[c] = b[c]
    fb = aff.build_features(aff.months_frame(months, r.household_size, r.num_dependants)).iloc[[-1]]
    fa = aff.build_features(aff.months_frame(follow, r.household_size, r.num_dependants)).iloc[[-1]]
    rows.append({"state_now": r.true_state, "score_now": aff._score(fb), "score_following": aff._score(fa),
                 "model_state_now": aff._state(fb), "model_state_following": aff._state(fa)})

v = pd.DataFrame(rows)
print(f"{len(v)} held-out households that are Strained or Critical now")
print("average score now:", round(v.score_now.mean()), "| following the budget:", round(v.score_following.mean()))
print("share whose score improves:", round((v.score_following > v.score_now).mean(), 2))
print("\nstate the model sees, now -> following the budget:")
print(pd.crosstab(v.model_state_now, v.model_state_following))

44 held-out households that are Strained or Critical now
average score now: 30 | following the budget: 70
share whose score improves: 0.91

state the model sees, now -> following the budget:
model_state_following  Critical  Stable  Strained
model_state_now                                  
Critical                      3       4        20
Stable                        0       0         2
Strained                      0       4        11


In [6]:
joblib.dump({"split_model": split_model, "one_size": ONE_SIZE, "x_cols": X_COLS,
             "variable": VARIABLE, "fixed": FIXED},
            ROOT / "backend" / "models" / "budget_split.joblib", compress=3)
print("saved:", (ROOT / "backend" / "models" / "budget_split.joblib").stat().st_size // 1024, "KB")


saved: 1801 KB
